**NOTE:** 
This is the same code as astro_trial_3_example.ipynb but calles solver from m_astrodf

i.e. 
`from simopt.solvers.m_astrodf import ASTRODF` instead of `from simopt.solvers.astrodf import ASTRODF`

#### Trial 1.1 

- Run M_ASTRODF on CNTNEWS-1 (a single objective problem.)

- **When error message on m_astrodf.py is commented**:  
    def solve(self, problem: Problem, ctx: Context) -> None:  
        self.problem = problem  
        # if problem.n_objectives != 2:  
        #     raise ValueError(f"M_ASTRODF needs two objective functions, but got {problem.n_objectives}.")  
        self._initialize_solving(ctx)`

    runs similar to astrodf.py 

- **When error message uncommented**:  
   prints ValueError. 

In [1]:
from simopt.solvers.m_astrodf import ASTRODF
from simopt.experiment_base import ProblemSolver

'''Run is a function defined in class ProblemSolver (check simopt--> experiment--> single.py). 
It runs the solver on the problem for a given number of macroreplications (n_macroreps)
'''

'''post_replicate runs post replications at solver's recommended solutions.
Also a function defined in class ProblemSolver.
CRN is also added here. '''

exp = ProblemSolver(solver_name="M_ASTRODF", problem_name="CNTNEWS-1")
exp.run(n_macroreps=2)
exp.post_replicate(n_postreps=50)
print("done")

print(type(exp.all_recommended_xs))
# print(exp.all_recommended_xs)
# print(exp.all_intermediate_budgets)

xs = getattr(exp, "all_recommended_xs")
budgets = getattr(exp, "all_intermediate_budgets")

"""Which solution was recommneded at which budget. Initialised in single.py"""
for m, (x_list, b_list) in enumerate(zip(xs, budgets)):
    print(f"macrorep {m}")
    for b, x in zip(b_list, x_list):
        print("  budget = ", b, ", x = ", x)
        
"""Eg: budget 1000 means solver spent its full budget of 1000 simulation replications, 
and its final incumbent x (solution) in macrorep 1 is x = 0.22473549788741493, 
in macrorep 0 is x = 0.20995000270146888. Each is a noisy estimate."""

print(type(exp.solver), type(exp.solver).__module__)
print(exp.solver.objective_type)
print(exp.check_compatibility())



ValueError: M_ASTRODF needs two objective functions, but got 1.

#### Trial 1.2 

- For new problem `CNTNEWS-2OBJ` defined on `cntnv.py`

- Runs fine when `use_gradients: True` which is the default. 

- Fails for `use_gradients: False`

In [2]:
from simopt.experiment_base import ProblemSolver

exp = ProblemSolver(
    solver_name="M_ASTRODF",
    problem_name="CNTNEWS-2OBJ",  # name you give the new problem
    solver_fixed_factors={"use_gradients": True},
)
print("n_objectives:", exp.problem.n_objectives)
print("compatibility:", repr(exp.check_compatibility()))  # expect ''
exp.run(n_macroreps=2, n_jobs=1)
print(exp.all_recommended_xs)
print(exp.all_intermediate_budgets)

n_objectives: 2
compatibility: ''
[[(0,), (0.3125,), (0.1875,), (0.22916666666666666,), (0.20833333333333334,), (0.21024068196614584,), (0.21024067741867233,), (0.21024066604998856,), (0.21024063762827913,), (0.21024056657400556,), (0.21024038893832162,), (0.21023994484911177,), (0.21023883462608714,), (0.21023605906852558,), (0.21022912017462167,), (0.2102117729398619,), (0.21016840485296248,), (0.21005998463571393,), (0.20992445936415324,), (0.20995834068204342,), (0.20994987035257087,), (0.20995119384155095,), (0.20995000270146888,), (0.20995000270146888,)], [(0,), (0.3125,), (0.23437500000000003,), (0.22216796875000003,), (0.22521972656250003,), (0.22516012191772464,), (0.22516011737025113,), (0.22516010600156736,), (0.22516007757985793,), (0.22516000652558435,), (0.2251598288899004,), (0.22515938480069056,), (0.22515827457766593,), (0.22515549902010437,), (0.22514856012620046,), (0.2251312128914407,), (0.2251047219382072,), (0.22507823098497368,), (0.22505174003174017,), (0.225025

#### Trial 1.3

- When `use_gradients: False` with all other conditions, violates the bound in construct_model.  
*Reason*: In `def get_coordinate_basis_interpolation_points`, `is_block_constraint` is true only for `sum(x_k)!=0`. When true: `clamp_with_epsilon` is not applied. With default initial solution set to 0, sum is also 0. Therefore, clamp is skipped. 

- Changed initial to 5.0

- *Note: `compatibility: ''` because `check_compatibility()` returns errors. Empty because no listed error messages.*



In [3]:
from simopt.experiment_base import ProblemSolver

exp = ProblemSolver(
    solver_name="M_ASTRODF",
    problem_name="CNTNEWS-2OBJ",  # name you give the new problem
    solver_fixed_factors={"use_gradients": False},
    problem_fixed_factors={"initial_solution": (5.0,)},
)
print("n_objectives:", exp.problem.n_objectives)
print("compatibility:", repr(exp.check_compatibility()))  #should be: ''
exp.run(n_macroreps=2, n_jobs=1)
print(exp.all_recommended_xs)
print(exp.all_intermediate_budgets)

n_objectives: 2
compatibility: ''
[[(5.0,), (0.01,), (0.2180660930392922,), (0.2059742570534784,), (0.2059742570534784,)], [(5.0,), (0.01,), (np.float64(0.45568441284402594),), (0.21288665505874413,), (0.1737909313852631,), (0.1737909313852631,)]]
[[5, 20, 112, 578, 1000], [5, 20, 94, 107, 421, 1000]]


Above codes extract solutions of CNTNEWS-1. 

Instead use the Quadnoiseproblem defined previously in "cf_trial_2_example.ipynb". The below code changes Quadnoiseproblem to have two objective functions:  
x^2+1, x^2+2  
Only x^2 + 1 is minimised. 

In [4]:
from __future__ import annotations

from typing import Annotated, ClassVar

import numpy as np
from pydantic import BaseModel, Field

from mrg32k3a.mrg32k3a import MRG32k3a
from simopt import dsl
from simopt.base import (
    ConstraintType,
    Model,
    Problem,
    VariableType,
)
from simopt.input_models import Normal
# from simopt.simulations.example import (
#     Example2ModelConfig,
#     ExampleModelConfig,
#     replicate,
#     replicate_discrete,
# )
from simopt.utils import override

# Like in simulation 
class QuadNoiseConfig2(BaseModel):
    x: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="decision variable"),
    ]
    noise_std: Annotated[
        float,
        Field(default=1e-8, ge=0.0, description="no noise standard deviation"),
    ]

class QuadNoiseModel2(Model[QuadNoiseConfig2]):
    """x^2 + 1 evaluated with tiny stochastic noise."""

    class_name_abbr: ClassVar[str] = "QUADNOISE2"
    class_name: ClassVar[str] = "x^2+1 and x^2+2"
    config_class: ClassVar[type[QuadNoiseConfig2]] = QuadNoiseConfig2
    n_rngs: ClassVar[int] = 1  ## number of random number generators for the model
    n_responses: ClassVar[int] = 2 ## no. of outputs from model (2 for 2 objectives)
    
    def __init__(self, fixed_factors: dict | None = None) -> None:
        super().__init__(fixed_factors)
        self.noise_model = Normal()
        
    def replicate(self, factors: QuadNoiseConfig2, rngs: list[MRG32k3a]) -> tuple[dict, dict]:
        x = np.asarray(factors.x, dtype=float)
        f1 = float(np.sum(x**2) + 1.0)
        f2 = float(np.sum(x**2) + 2.0)

        # Tiny noise: set noise_std = 0.0 for exact deterministic behavior.
        noise_f1 = self.noise_model.random(rngs[0], mu=0.0, sigma=factors.noise_std) # Add noise for both objectives.
        noise_f2 = self.noise_model.random(rngs[0], mu=0.0, sigma=factors.noise_std)
        noisy_value1 = f1 + noise_f1
        noisy_value2 = f2 + noise_f2

        responses = {"est_f1(x)": noisy_value1, "No change in f2(x)": noisy_value2} #
        gradients = {"est_f1(x)": {"x": 2.0 * x}, "No change in f2(x)": {"x": 2.0 * x}} 
        return responses, gradients
    
    #Similar to example.py in models.   
class QuadNoiseProblemConfig2(BaseModel):
    initial_solution: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="initial point"),
    ]
    budget: Annotated[
        int,
        Field(default=1000, gt=0, description="max simulation budget"),
    ]
    
class QuadNoiseProblem2(Problem):
    """Minimize x^2 + 1 under noisy stochastic replications."""

    class_name_abbr: ClassVar[str] = "QUADNOISE"
    class_name: ClassVar[str] = "Min x^2+1 while taking in x^2+2"
    config_class: ClassVar[type[BaseModel]] = QuadNoiseProblemConfig2
    model_class: ClassVar[type[Model]] = QuadNoiseModel2
    n_objectives: ClassVar[int] = 2 # 2 objectives
    n_stochastic_constraints: ClassVar[int] = 0
    minmax: ClassVar[tuple[int, ...]] = (-1, -1) # length equal to n_objectives
    constraint_type: ClassVar[ConstraintType] = ConstraintType.UNCONSTRAINED
    variable_type: ClassVar[VariableType] = VariableType.CONTINUOUS
    gradient_available: ClassVar[bool] = True
    model_default_factors: ClassVar[dict] = {}
    model_decision_factors: ClassVar[set[str]] = {"x"}
    
    @property
    def optimal_value(self) -> float | None:
        return 1.0 # min of x^2 + 1 (only)
    
    @property
    def optimal_solution(self) -> tuple | None:
        return (0.0,)
    
    @override
    def build(self) -> dsl.Model:
        problem = dsl.Model()
        x0 = self.factors["initial_solution"]
        x = problem.add_continuous_vector(
            lb=-10.0, ub=10.0, shape=(len(x0),), initial=x0,
        )
        simulation = self.add_simulation(problem, {"x": x})
        problem.minimize(dsl.mean(simulation.metric("est_f1(x)"))) 
        problem.minimize(dsl.mean(simulation.metric("No change in f2(x)"))) 
        return problem
    
    def vector_to_factor_dict(self, vector: tuple) -> dict:
        return {"x": vector[:]}
    
    def get_random_solution(self, rand_sol_rng: MRG32k3a) -> tuple:
        # unchanged from the example
        return tuple(
            rand_sol_rng.mvnormalvariate(
                mean_vec=[0.0] * self.dim,
                cov=np.eye(self.dim).tolist(),
                factorized=False,
            )
        )

In [5]:
from simopt.experiment_base import ProblemSolver

exp = ProblemSolver(
    solver_name="M_ASTRODF",
    problem=QuadNoiseProblem2(fixed_factors={"initial_solution": (5.0,), "budget": 1000}),
    solver_fixed_factors={"use_gradients": False},
)
print("compatibility:", repr(exp.check_compatibility()))
exp.run(n_macroreps=2, n_jobs=1)
print(exp.all_recommended_xs)
print(exp.all_intermediate_budgets)

compatibility: ''
[[(5.0,), (0.0,), (0.0,)], [(5.0,), (0.0,), (0.0,)]]
[[5, 30, 1000], [5, 30, 1000]]


In [9]:
import logging
logging.basicConfig(level=logging.DEBUG, force=True)
from simopt.experiment_base import ProblemSolver

exp = ProblemSolver(
    solver_name="M_ASTRODF",
    problem=QuadNoiseProblem2(fixed_factors={"initial_solution": (5.0,), "budget": 1000}),
    solver_fixed_factors={"use_gradients": False},
)
print("compatibility:", repr(exp.check_compatibility()))
exp.run(n_macroreps=2, n_jobs=1)
print(exp.all_recommended_xs)
print(exp.all_intermediate_budgets)

INFO:root:Running solver M_ASTRODF on problem QUADNOISE.
DEBUG:root:Starting macroreplications
DEBUG:root:Macroreplication 0: starting solver M_ASTRODF on problem QUADNOISE.
DEBUG:root:iter 1: delta_k=10.0 incumbent_x=(5.0,) candidate_x=(-5.0,) rho=[0.]
DEBUG:root:iter 2: delta_k=5.0 incumbent_x=(5.0,) candidate_x=(0.0,) rho=[1.]
DEBUG:root:iter 3: delta_k=4.979057727189805 incumbent_x=(0.0,) candidate_x=(0.0,) rho=0
DEBUG:root:iter 4: delta_k=2.4895288635949027 incumbent_x=(0.0,) candidate_x=(0.0,) rho=0
DEBUG:root:iter 5: delta_k=1.2447644317974513 incumbent_x=(0.0,) candidate_x=(0.0,) rho=0
DEBUG:root:iter 6: delta_k=0.6223822158987257 incumbent_x=(0.0,) candidate_x=(0.0,) rho=0
DEBUG:root:iter 7: delta_k=0.31119110794936283 incumbent_x=(0.0,) candidate_x=(0.0,) rho=0
DEBUG:root:iter 8: delta_k=0.15559555397468142 incumbent_x=(0.0,) candidate_x=(0.0,) rho=0
DEBUG:root:iter 9: delta_k=0.07779777698734071 incumbent_x=(0.0,) candidate_x=(0.0,) rho=0
DEBUG:root:iter 10: delta_k=0.038898

compatibility: ''
[[(5.0,), (0.0,), (0.0,)], [(5.0,), (0.0,), (0.0,)]]
[[5, 30, 1000], [5, 30, 1000]]


In [12]:
# Note: For this to run above QuadNoise code should be defined. Or imported. 

import numpy as np
import pandas as pd
from simopt.solvers.m_astrodf import ASTRODF
from simopt.experiment_base import ProblemSolver

LOG = []
STATE = {"mrep": -1}  # to identify which macroreplication.


# Defining a class that inherits everything from astrodf.

class LoggingASTRODF(ASTRODF):

    def _initialize_solving(self, ctx):
        super()._initialize_solving(ctx) #run original setup unchanged
        STATE["mrep"] += 1 #for macrorep (each macrorep call initialize solving once)

    def construct_model(self, ctx):
        delta_in = self.delta_k # TR radius before step
        out = super().construct_model(ctx) # run original MU step and store output.
        fval, y_var, q, grad, hessian, solns = out
        
        '''outputs: fval = estimate at each point in mathcal{X}_k
        y_var = points in mathcal{X}_k (from get_rotated_basis_interpolation_points in astrodf.py)
        q = from get_model_coefficients,
        grad, hessian = sliced out from q
        solns = interpolation_solns from construct_model (function estimates of interpolation points are 
        appended to this empty list in def iterates as "interpolation_solns.append(adapt_soln)")
        '''

        m = 2 * self.problem.dim + 1 # No. of interpolation points
        
        '''In the next loop [-m] because construct_model repeats its inner loop 
        until criticality check satisfied. 
        LOG empty list stores all the infor. Each becomes a column. 
        mrep=macrorep number
        iter=iteration number 
        delta_in = initial delta at the start of iteration
        incumbent_x = current centre (X_k^0 from Alg 3.1)
        point_idx = i (=0,1,2 for 1D case, 0 is incumbent_x)
        x = interpolation point location
        F_hat = estimate bar{F}_k^i(N_k^i) using sample size N_k^i
        sample_size = sample size N_k^i
        var_F = sample variance of the objective at that point.
        model_grad, model_hessian_diag = fitted local model
        '''
        
        for i, s in enumerate(solns[-m:]):
            LOG.append({
                "mrep": STATE["mrep"],
                "iter": self.iteration_count,
                "delta_in": delta_in,
                "incumbent_x": tuple(np.ravel(self.incumbent_x)),
                "point_idx": i,
                "x": tuple(np.ravel(s.x)),
                "F_hat": float(np.ravel(fval[i])[0]),
                "sample_size": int(s.n_reps),
                "var_F": float(np.ravel(s.objectives_var)[0]),
                "model_grad": tuple(np.ravel(grad)),
                "model_hessian_diag": tuple(np.ravel(hessian)),
            })
        return out

# set solver to not use gradients even though gradient can be calculated here. 
# budget for replication set to 1000 (similar to "CNTNEWS-1")
# Start with initial = 5.0

solver = LoggingASTRODF(fixed_factors={"use_gradients": False})
problem = QuadNoiseProblem(fixed_factors={"initial_solution": (5.0,), "budget": 1000})

exp = ProblemSolver(solver=solver, problem=problem)
exp.run(n_macroreps=2, n_jobs=1) #n_jobs = 1 to run sequentially.

df = pd.DataFrame(LOG)
pd.set_option("display.width", 250, "display.max_rows", None, "display.max_columns", None)
print(df[["mrep", "iter", "delta_in", "point_idx", "x", "F_hat", "sample_size", "model_grad", "model_hessian_diag"]])
df.to_csv("astrodf_design_sets_quad.csv", index=False)

ValueError: M_ASTRODF needs two objective functions, but got 1.

In [6]:
# Note: For this to run above QuadNoise code should be defined. Or imported. 

import numpy as np
import pandas as pd
from simopt.solvers.m_astrodf import ASTRODF
from simopt.experiment_base import ProblemSolver

LOG = []
STATE = {"mrep": -1}  # to identify which macroreplication.
STEP_LOG = []   # for SM and TM steps of Alg 3.1

# Defining a class that inherits everything from astrodf.

class LoggingASTRODF(ASTRODF):

    def _initialize_solving(self, ctx):
        super()._initialize_solving(ctx) #run original setup unchanged
        STATE["mrep"] += 1 #for macrorep (each macrorep call initialize solving once)

    def construct_model(self, ctx):
        delta_in = self.delta_k # TR radius before step
        out = super().construct_model(ctx) # run original MU step and store output.
        fval, y_var, q, grad, hessian, solns = out
        
        '''outputs: fval = estimate at each point in mathcal{X}_k
        y_var = points in mathcal{X}_k (from get_rotated_basis_interpolation_points in astrodf.py)
        q = from get_model_coefficients,
        grad, hessian = sliced out from q
        solns = interpolation_solns from construct_model (function estimates of interpolation points are 
        appended to this empty list in def iterates as "interpolation_solns.append(adapt_soln)")
        '''
        
        self._delta_sm = self.delta_k  # Delta used in SM

        m = 2 * self.problem.dim + 1 # No. of interpolation points
        
        '''In the next loop [-m] because construct_model repeats its inner loop 
        until criticality check satisfied. 
        LOG empty list stores all the infor. Each becomes a column. 
        mrep=macrorep number
        iter=iteration number 
        delta_in = initial delta at the start of iteration
        incumbent_x = current centre (X_k^0 from Alg 3.1)
        point_idx = i (=0,1,2 for 1D case, 0 is incumbent_x)
        x = interpolation point location
        F_hat = estimate bar{F}_k^i(N_k^i) using sample size N_k^i
        sample_size = sample size N_k^i
        var_F = sample variance of the objective at that point.
        model_grad, model_hessian_diag = fitted local model
        '''
        
        for i, s in enumerate(solns[-m:]):
            LOG.append({
                "mrep": STATE["mrep"],
                "iter": self.iteration_count,
                "delta_in": delta_in,
                "incumbent_x": tuple(np.ravel(self.incumbent_x)),
                "point_idx": i,
                "x": tuple(np.ravel(s.x)),
                "F_hat": float(np.ravel(fval[i])[0]),
                "sample_size": int(s.n_reps),
                "var_F": float(np.ravel(s.objectives_var)[0]),
                "model_grad": tuple(np.ravel(grad)),
                "model_hessian_diag": tuple(np.ravel(hessian)),
            })
        return out
    
    def iterate(self, ctx):
        x_before = np.ravel(np.array(self.incumbent_x, dtype=float)) #saves the incumbent_x before def iterate starts
        super().iterate(ctx)  # from original astrodf.py

        # from visited_pts_list
        cand_x = np.ravel(np.array(self.visited_pts_list[-1].x, dtype=float))
        x_after = np.ravel(np.array(self.incumbent_x, dtype=float)) # incumbent_x after iterates

        step = cand_x - x_before #compare before and after
        accepted = not np.allclose(x_after, x_before) # accepted = True if incumbent_x changed.
        from_pattern_search = accepted and not np.allclose(x_after, cand_x) #astrodf code swaps best interpolation point
        delta_sm = getattr(self, "_delta_sm", np.nan) 
        step_norm = float(np.linalg.norm(step)) # length of step

        STEP_LOG.append({
            "mrep": STATE["mrep"],
            "iter": self.iteration_count,
            "incumbent_before": tuple(x_before),
            "delta_sm": delta_sm,
            "candidate_x": tuple(cand_x),
            "step": tuple(step),
            "step_norm": step_norm,
            "accepted": accepted,
            "from_pattern_search": from_pattern_search,
            "incumbent_after": tuple(x_after),
            "delta_next": self.delta_k,
        })

# set solver to not use gradients even though gradient can be calculated here. 
# budget for replication set to 1000 (similar to "CNTNEWS-1")
# Start with initial = 5.0

solver = LoggingASTRODF(fixed_factors={"use_gradients": False})
problem = QuadNoiseProblem(fixed_factors={"initial_solution": (5.0,), "budget": 1000})

exp = ProblemSolver(solver=solver, problem=problem)
exp.run(n_macroreps=2, n_jobs=1) #n_jobs = 1 to run sequentially.

df = pd.DataFrame(LOG)
steps = pd.DataFrame(STEP_LOG)
pd.set_option("display.width", 250, "display.max_rows", None, "display.max_columns", None)

print(steps)
steps.to_csv("astrodf_steps_quad.csv", index=False)

merged = df.merge(
    steps[["mrep", "iter", "delta_sm", "step", "accepted", "from_pattern_search", "delta_next"]],
    on=["mrep", "iter"],
)

print(merged[["mrep", "iter", "delta_in", "point_idx", "x", "F_hat", "sample_size",
            "delta_sm", "step", "accepted"]])
merged.to_csv("astrodf_design_sets_quad.csv", index=False)

    mrep  iter incumbent_before      delta_sm candidate_x      step  step_norm  accepted  from_pattern_search incumbent_after    delta_next
0      0     1           (5.0,)  1.000000e+01     (-5.0,)  (-10.0,)       10.0     False                False          (5.0,)  5.000000e+00
1      0     2           (5.0,)  5.000000e+00      (0.0,)   (-5.0,)        5.0      True                False          (0.0,)  6.891429e+00
2      0     3           (0.0,)  4.979058e+00      (0.0,)    (0.0,)        0.0     False                False          (0.0,)  2.489529e+00
3      0     4           (0.0,)  2.489529e+00      (0.0,)    (0.0,)        0.0     False                False          (0.0,)  1.244764e+00
4      0     5           (0.0,)  1.244764e+00      (0.0,)    (0.0,)        0.0     False                False          (0.0,)  6.223822e-01
5      0     6           (0.0,)  6.223822e-01      (0.0,)    (0.0,)        0.0     False                False          (0.0,)  3.111911e-01
6      0     7      